# Retrieve metadata for a mapping process

This tutorial shows how pysdmx can retrieve the structures required to map data from one structure to another.

## Step 1: Connect to the service that contains the required structures

The client can connect to any compatible SDMX-RESTv2 service.

In [3]:
from pathlib import Path

from pysdmx.api.fmr import RegistryClient
from pysdmx.io import read_sdmx

client = RegistryClient("https://registry.sdmx.org/sdmx/v2")

## Step 2: Retrieve the required mapping definition and inspect its content

In [4]:
representation_map = client.get_code_map("SDMX", "CL_AREA_ISO_M49", "1.0")
print(representation_map)
for vm in representation_map.maps:
    print(f"{vm.source} => {vm.target}")

id: CL_AREA_ISO_M49, name: ISO to M49 codes, version: 1.0, agency: SDMX, source: urn:sdmx:org.sdmx.infomodel.codelist.Codelist=SDMX:CL_AREA(2.0), target: urn:sdmx:org.sdmx.infomodel.codelist.Codelist=SDMX:CL_AREA(2.0), maps: 247 valuemaps
AD => 020
AE => 784
AF => 004
AG => 028
AI => 660
AL => 008
AM => 051
AO => 024
AR => 032
AS => 016
AT => 040
AU => 036
AW => 533
AX => 248
AZ => 031
BA => 070
BB => 052
BD => 050
BE => 056
BF => 854
BG => 100
BH => 048
BI => 108
BJ => 204
BL => 652
BM => 060
BN => 096
BO => 068
BQ => 535
BR => 076
BS => 044
BT => 064
BV => 074
BW => 072
BY => 112
BZ => 084
CA => 124
CC => 166
CD => 180
CF => 140
CG => 178
CH => 756
CI => 384
CK => 184
CL => 152
CM => 120
CN => 156
CO => 170
CR => 188
CU => 192
CV => 132
CW => 531
CX => 162
CY => 196
CZ => 203
DE => 276
DJ => 262
DK => 208
DM => 212
DO => 214
DZ => 012
EC => 218
EE => 233
EG => 818
EH => 732
ER => 232
ES => 724
ET => 231
FI => 246
FJ => 242
FK => 238
FM => 583
FO => 234
FR => 250
GA => 266
GB => 826
G

This is a very simple mapping between two-letter ISO country codes and their corresponding M49 codes.

## Step 3: Load a data file

Note: The CSV file was generated automatically, so there is no guarantee that the data are meaningful.

In [5]:
file_path = Path("../samples/bpm6_bop_m.csv")
if not file_path.exists():
    file_path = Path("docs/tutorials/samples/bpm6_bop_m.csv")
submission = read_sdmx(file_path)

## Step 5: Inspect the data

The data are represented as Pandas DataFrames.

In [6]:
for ds in submission.data:
    display(ds.data)

,FREQ,ADJUSTMENT,REF_AREA,COUNTERPART_AREA,REF_SECTOR,COUNTERPART_SECTOR,FLOW_STOCK_ENTRY,ACCOUNTING_ENTRY,INT_ACC_ITEM,FUNCTIONAL_CAT,...,TITLE,UNIT_MULT,DECIMALS,TIME_PER_COLLECT,REF_PERIOD_DETAIL,COMPILING_ORG,DATA_COMP,DISS_ORG,TITLE_COMPL,OBS_VALUE
0,M,N,HU,1A,S1,S11,K7,A,CA,D,...,,6,0,S,,,,,,100.0
1,M,N,GR,1A,S1,S11,K7,A,CA,D,...,,6,0,S,,,,,,102.5
2,M,N,R1,1A,S1,S11,K7,A,CA,D,...,,6,0,S,,,,,,98.75
3,Q,N,R1,1A,S1,S11,K7,A,CA,D,...,,6,0,S,,,,,,110.0


## Step 6: Map the `REF_AREA` and `COUNTERPART_AREA` columns

This should be done more subtly by getting the DSD and inferring which components can be mapped. For this demonstration, explicitly selecting the columns is sufficient to show how mapping works. pysdmx will soon offer a `map()` function.

In [7]:
mapping_candidates = ["REF_AREA", "COUNTERPART_AREA"]
code_map = {vm.source: vm.target for vm in representation_map}

for ds in submission.data:
    df = ds.data
    for component in mapping_candidates:
        df[component] = df[component].replace(code_map)

## Step 7: Inspect the mapped data

Check that `GR` (Greece) and `HU` (Hungary) have been mapped to `300` and `348`, respectively. `R1` (Belgo-Luxembourg Economic Union) remains unchanged because it is not in the representation map.

In [9]:
for ds in submission.data:
    display(ds.data[["REF_AREA"]].drop_duplicates())

,REF_AREA
0,348
1,300
2,R1
